# 第 11 章: 評価指標と交差検証の探索と可視化

Survived の決定木について、混同行列と、交差検証の分割ごとのスコアを確認する。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: FSharp.Stats, 0.6.0"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter02.IrisPreprocessing
open MachineLearning.Chapter11.CrossValidation
open MachineLearning.Chapter11.Datasets
open MachineLearning.Chapter11.Experiments
open MachineLearning.Chapter11.Metrics

let x, t = SurvivedCsv.Load(Path.Combine(dataDir (), "Survived.csv")) |> prepareSurvived

## 混同行列

In [ ]:
let split = splitTrainTest 0.3 0 x t
let predicted = decisionTree split.XTrain split.TTrain split.XTest
let cm = confusionMatrix Survived split.TTest predicted
cm

In [ ]:
Chart.Heatmap(
    zData = [ [ cm.TN; cm.FP ]; [ cm.FN; cm.TP ] ],
    X = [ "死亡と予測"; "生存と予測" ],
    Y = [ "実際は死亡"; "実際は生存" ]
)
|> Chart.withTitle "混同行列（テストデータ）"

In [ ]:
[| {| 適合率 = precision cm; 再現率 = recall cm; F値 = f1Score cm |} |]

## 交差検証の分割ごとのスコア

In [ ]:
let folds = kFold NSplits Seed x.Length

let foldScores =
    SurvivedMetrics
    |> List.map (fun (name, metric) -> name, crossValidate decisionTree metric folds x t |> Seq.toList)

foldScores
|> List.map (fun (name, scores) -> Chart.Point(x = [ 1 .. scores.Length ], y = scores, Name = name))
|> Chart.combine
|> Chart.withTitle "分割ごとのスコア（決定木、深さ 2）"
|> Chart.withXAxisStyle "分割"
|> Chart.withYAxisStyle "スコア"

In [ ]:
foldScores
|> List.map (fun (name, scores) -> {| 指標 = name; 最小 = List.min scores; 最大 = List.max scores |})
|> List.toArray